### Tools

models can requeset to call tools that perform tast such that fetching data from database, searching the web, or running code.Tools are paring of

  1. A schema, including the name of tools , a discription ,and/or argument difnitions       (often a  JSON schema).
   
  2. A function or coroutine to execute.

In [1]:

import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model =init_chat_model("groq:llama-3.1-8b-instant")

response=model.invoke("write me an essay on AI")
response 

AIMessage(content='**The Rise of Artificial Intelligence: Opportunities and Challenges**\n\nArtificial intelligence (AI) has become a ubiquitous term in modern technology, with applications in various sectors, including healthcare, finance, transportation, and education. The rapid advancement of AI has transformed the way we live, work, and interact with each other. In this essay, we will explore the opportunities and challenges presented by AI, and examine the potential impact of this technology on society.\n\n**The Benefits of AI**\n\nOne of the most significant benefits of AI is its ability to automate tasks, freeing humans from mundane and repetitive work. This has led to increased productivity, efficiency, and accuracy in various industries. For instance, AI-powered chatbots have revolutionized customer service, providing 24/7 support and assistance to customers. In healthcare, AI has improved diagnosis accuracy and enabled personalized medicine, leading to better patient outcomes

In [5]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at the location"""
    return f"It's sunny in {location}"

model_with_tool = model.bind_tools([get_weather])

In [7]:
response=model_with_tool.invoke("what is the Weather in the Banglore ")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'y6mjk9pew', 'function': {'arguments': '{"location":"Banglore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.033571973, 'completion_tokens_details': None, 'prompt_time': 0.024677267, 'prompt_tokens_details': None, 'queue_time': 0.048989189, 'total_time': 0.05824924}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019eb816-e64b-7ff0-b89e-feb5ec5c189f-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Banglore'}, 'id': 'y6mjk9pew', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236}


In [8]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Banglore'},
  'id': 'y6mjk9pew',
  'type': 'tool_call'}]

### Tool Execution Loop

In [24]:
#Model generstes tool call 
messages=[{"role":"user","content":"What's the weather in Bangalore?"}]
ai_msg=model_with_tool.invoke(messages)
messages.append(ai_msg)

#Execute tools and collect results 
for tool_call in ai_msg.tool_calls:
    #execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call["args"])
    messages.append(tool_result)
 


 #pass result back to model for the final result 
final_response = model_with_tool.invoke(messages)
 
print(final_response.content)
 

It's sunny in Bangalore


In [25]:
messages

[{'role': 'user', 'content': "What's the weather in Bangalore?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7dzy9h8jx', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.027284908, 'completion_tokens_details': None, 'prompt_time': 0.013751895, 'prompt_tokens_details': None, 'queue_time': 0.102328229, 'total_time': 0.041036803}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb835-ba87-7811-a31c-276bc5ea484d-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '7dzy9h8jx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}),
 "It's sunny in Bang

In [26]:
model_with_tool

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x72ab743e2ea0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x72ab7422e660>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at the location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])